# Tier 3: Molecular Modeling & Force Fields
## Lab 2.1 — Solvation Models, Counter-Ion Condensation, and Molecular Dynamics System Preparation

Welcome to **Lab 2.1** of the *Computational Structural Biology and Generative Macromolecular Design* curriculum!

### Why This Lab Matters
In Lab 2.0, we explored the classical Class I potential energy function ($V_{\text{bond}}, V_{\text{angle}}, V_{\text{dihedral}}, V_{\text{vdW}}, V_{\text{elec}}$) and energy-minimized *Ideonella sakaiensis* PETase in implicit solvent. However, in nature, biomacromolecules do not operate in isolation—they are enveloped by a dynamic solvent sheath of water molecules and physiological electrolyte ions.

Before executing molecular dynamics (MD) trajectories, bioinformaticians must master **system preparation**:
1. **Implicit vs. Explicit Solvation Mechanics:** When is generalized Born (GB/OBC) implicit solvent appropriate (e.g., high-throughput conformational sampling, binding free energy MM/PBSA), and why is explicit water (TIP3P, TIP4P-Ew, OPC) mandatory when investigating directional hydrogen-bonding networks, bridging waters, and hydrophobic dewetting transitions?
2. **Water Model Stereochemistry:** Rigid 3-site models (TIP3P, SPC/E) vs. 4-site models (TIP4P-Ew, OPC) featuring an off-atom charge center (dummy atom $M$) that reproduces liquid water's quadrupole moment and bulk density.
3. **Manning Counter-Ion Condensation & Neutralization:** Unneutralized charged solutes simulated with Particle Mesh Ewald (PME) electrostatics generate infinite lattice energy artifacts. For polyanionic nucleic acids (e.g., RNA phosphodiester backbones), counter-ions undergo Manning condensation, forming an electrostatic screening atmosphere.
4. **Physiological Ionic Strength Contract:** In accordance with standard biological biophysics, simulation systems must be neutralized and brought to **$150\,\text{mM } \text{NaCl}$** physiological salt concentration.
5. **Periodic Boundary Conditions (PBC):** Constructing rhombic dodecahedron or cubic periodic boxes with a mandatory minimum solute-box edge clearance ($d_{\text{buffer}} \ge 1.0\,\text{nm} = 10.0\,\text{\AA}$) to prevent self-interaction artifacts across periodic boundaries.

---

### Learning Objectives
By the end of this laboratory, you will be able to:
- Formulate the physical trade-offs between Poisson-Boltzmann/Generalized Born implicit solvent and explicit multi-site water models.
- Contrast the geometric and electrostatic architectures of 3-site (TIP3P) versus 4-site (TIP4P-Ew, OPC) water models.
- Calculate exact counter-ion and co-ion counts to achieve total system charge neutrality and $150\,\text{mM } \text{NaCl}$ ionic strength.
- Construct cubic periodic simulation cells using OpenMM's `app.Modeller`, solvating the protein with explicit TIP3P water and adding neutralizing $\text{Na}^+$ and $\text{Cl}^-$ ions.
- Parameterize and solvate the polyanionic purine riboswitch (PDB: `1Y26`), exploring counter-ion condensation along phosphodiester backbones.
- Execute a production-grade initial relaxation protocol (Steepest Descent + L-BFGS) in explicit water, verifying box dimensions and density.
- Self-assess your mastery using interactive `jupyterquiz` questions.

---

### Running Systems in Tier 3
- **Protein System:** *Ideonella sakaiensis* **PETase** (PDB: `6ANE`), solvated in a periodic box with $150\,\text{mM } \text{NaCl}$ to prepare for production MD.
- **Nucleic Acid System:** *Vibrio vulnificus* *add* **A-riboswitch** (PDB: `1Y26`), analyzing high-density $\text{Na}^+$ and $\text{Mg}^{2+}$ ion condensation around negative phosphodiester diesters.

Let's begin by configuring our computational environment.


### Section 0: Environment Setup and Pinned Dependencies
We pin all core simulation, parsing, and visualization packages to ensure reproducible execution across Google Colab runtimes.


In [ ]:
# Install required scientific simulation and bioinformatics packages
!pip install -q openmm>=8.2.0 pdbfixer biopython==1.83 jupyterquiz==2.8.1 py3Dmol==2.1.0 numpy pandas scipy matplotlib requests

import os
import sys
import math
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import py3Dmol
from Bio.PDB import PDBParser, PDBIO, Select
from pdbfixer import PDBFixer

# OpenMM imports
import openmm as mm
from openmm import app
from openmm import unit
from IPython.display import display

# Pinned course reproducibility seed
COURSE_SEED = 42
random.seed(COURSE_SEED)
np.random.seed(COURSE_SEED)

os.makedirs("data", exist_ok=True)
os.makedirs("manifests", exist_ok=True)

print("Lab 2.1 environment initialized successfully.")
print(f"Python version: {sys.version.split()[0]}")
print(f"OpenMM version: {mm.__version__} | BioPython version: 1.83")
print(f"Reproducibility seed: {COURSE_SEED}")


---
## Section 1: The Physics of Solvation — Implicit vs. Explicit Models

A folded biomacromolecule in solution experiences non-covalent interactions with surrounding water:
$$\Delta G_{\text{solv}} = \Delta G_{\text{cavity}} + \Delta G_{\text{vdW}} + \Delta G_{\text{elec}}$$

In computational biophysics, two primary paradigms exist for handling this solvent:

| Feature / Property | Implicit Solvent (GB / PB) | Explicit Solvent (TIP3P / TIP4P-Ew / OPC) |
|---|---|---|
| **Representation** | Structureless dielectric continuum ($\epsilon_r \approx 78.4$) | All-atom discrete water molecules with fixed geometry |
| **Computational Cost** | Very fast (no solvent degrees of freedom) | Slower ($>80\%$ of system atoms are solvent) |
| **Degrees of Freedom** | Solute atoms only ($3N_{\text{solute}}$) | Solute $+$ thousands of waters ($3N_{\text{solute}} + 3N_{\text{water}}$) |
| **Viscosity & Friction** | Lacks hydrodynamic drag; requires stochastic Langevin thermostat | Natural solvent viscosity, Brownian collision, and hydrodynamic friction |
| **Directional H-Bonds** | Cannot form discrete, oriented water bridges | Accurately models structured hydration shells and bridging waters |
| **Hydrophobic Effect** | Approximated via surface area terms ($\gamma \cdot \text{SASA}$) | Arises naturally from water-water entropy and hydrogen-bond network disruption |

### Anatomy of Multi-Site Explicit Water Models
1. **3-Site Models (TIP3P, SPC/E):** Three interaction sites positioned directly on the atomic nuclei (one Oxygen atom with charge $-2q$, two Hydrogen atoms each with charge $+q$). Fast to compute via rigid constraint algorithms (SETTLE, SHAKE).
2. **4-Site Models (TIP4P-Ew, OPC):** Move the negative charge off the Oxygen nucleus to a massless dummy site (termed the **$M$-site**) shifted along the $\text{H}-\text{O}-\text{H}$ bisector toward the Hydrogens. This geometry significantly improves the electrostatic quadrupole moment, dielectric constant, and temperature-density profile.

Let's compute the physical charge layout and dipole moment of standard TIP3P vs. 4-site water models.


In [ ]:
# Physical comparison of explicit water model geometries
WATER_MODELS = {
    "TIP3P": {
        "description": "Transferable Intermolecular Potential 3-Point (Jorgensen, 1983)",
        "sites": 3,
        "r_OH_angstrom": 0.9572,
        "angle_HOH_deg": 104.52,
        "q_O_elem": -0.834,
        "q_H_elem": +0.417,
        "dummy_M_offset_angstrom": 0.0,
        "dielectric_constant": 82.0
    },
    "TIP4P-Ew": {
        "description": "TIP4P Ewald-Optimized 4-Point (Horn et al., 2004)",
        "sites": 4,
        "r_OH_angstrom": 0.9572,
        "angle_HOH_deg": 104.52,
        "q_O_elem": 0.0,  # Oxygen holds only Lennard-Jones parameters
        "q_H_elem": +0.52422,
        "q_M_elem": -1.04844,  # Massless M-site holds negative charge
        "dummy_M_offset_angstrom": 0.125,
        "dielectric_constant": 73.0
    },
    "OPC": {
        "description": "Optimal Point Charge 4-Point (Izadi et al., 2014)",
        "sites": 4,
        "r_OH_angstrom": 0.8724,
        "angle_HOH_deg": 103.60,
        "q_O_elem": 0.0,
        "q_H_elem": +0.6791,
        "q_M_elem": -1.3582,
        "dummy_M_offset_angstrom": 0.1594,
        "dielectric_constant": 78.4
    }
}

water_df = pd.DataFrame([
    {
        "Model": k,
        "Sites": v["sites"],
        "r_OH (Å)": v["r_OH_angstrom"],
        "Angle (°)": v["angle_HOH_deg"],
        "q_H (e)": v["q_H_elem"],
        "Negative Site": "O" if v["sites"] == 3 else "M (Dummy)",
        "Dielectric ε": v["dielectric_constant"],
        "Citation": v["description"]
    }
    for k, v in WATER_MODELS.items()
])

print("Table 2.1: Multi-Site Classical Water Models in Molecular Simulation")
display(water_df)


### Formative Assessment: Water Model Selection
Test your comprehension of explicit and implicit solvation physics.


In [ ]:
from jupyterquiz import display_quiz

quiz_section_1 = [
    {
        "question": "Why does a 4-site water model (such as TIP4P-Ew or OPC) place its negative partial charge on a massless dummy M-site rather than directly on the oxygen nucleus?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Shifting the negative charge along the H-O-H bisector improves the gas-to-liquid electrostatic quadrupole moment, reproducing bulk water density and dielectric properties much more accurately than a 3-site model.",
                "correct": True,
                "feedback": "Correct! The M-site displacement produces an electrostatic potential that mimics real quantum mechanical charge distribution, improving dielectric and structural fidelity."
            },
            {
                "answer": "Because oxygen nuclei have an electrical charge of zero in quantum mechanics.",
                "correct": False,
                "feedback": "Incorrect. Oxygen has 8 protons; the M-site is an empirical classical parametrization trick."
            },
            {
                "answer": "To allow the water molecule to undergo spontaneous covalent dissociation into hydronium (H3O+) and hydroxide (OH-).",
                "correct": False,
                "feedback": "Incorrect. Standard classical water models are rigid and non-reactive."
            },
            {
                "answer": "Because OpenMM is incapable of calculating Lennard-Jones forces on oxygen atoms.",
                "correct": False,
                "feedback": "Incorrect. OpenMM evaluates non-bonded forces on oxygen in all water models."
            }
        ]
    },
    {
        "question": "Under what specific biological modeling condition is generalized Born (GB) implicit solvent NOT recommended?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "When investigating structural bridging waters in an enzyme active site or hydrophobic dewetting transitions inside a ligand-binding cavity.",
                "correct": True,
                "feedback": "Correct! Continuum dielectric models have no discrete molecular water positions and cannot form oriented water-mediated hydrogen bonds."
            },
            {
                "answer": "When performing fast conformational sampling of a small disordered peptide.",
                "correct": False,
                "feedback": "Incorrect. Fast sampling of peptides is a classic, highly effective use case for implicit solvent."
            },
            {
                "answer": "When running end-point binding free energy calculations via MM/PBSA.",
                "correct": False,
                "feedback": "Incorrect. MM/PBSA explicitly relies on continuum solvent post-processing."
            },
            {
                "answer": "When the simulation temperature is set to 300 Kelvin.",
                "correct": False,
                "feedback": "Incorrect. Implicit solvent operates at any physiological temperature."
            }
        ]
    }
]

display_quiz(quiz_section_1)


---
## Section 2: Counter-Ion Neutralization and Physiological Ionic Strength

Biological macromolecules in vivo exist in an electrolyte solution containing monovalent ($\text{Na}^+, \text{K}^+, \text{Cl}^-$) and divalent ($\text{Mg}^{2+}, \text{Ca}^{2+}$) salts.

### 1. Mandatory Neutralization for Periodic Boundary Conditions (PME)
In modern explicit-solvent molecular dynamics, long-range electrostatic interactions are computed using **Particle Mesh Ewald (PME)** summation.
PME splits the $1/r$ Coulomb potential into a short-range real-space sum and a long-range reciprocal-space Fourier transform.
**Mathematical Constraint:** PME mathematically requires that the **net charge of the periodic unit cell equals exactly zero**:
$$Q_{\text{net}} = \sum_{i \in \text{box}} q_i = 0$$
If a net charge ($Q_{\text{net}} \ne 0$) is simulated under PME, the background neutralizing plasma correction introduces an artificial energy artifact proportional to $Q_{\text{net}}^2 / L_{\text{box}}$, which distorts conformational equilibria.

### 2. Manning Counter-Ion Condensation along Polyanionic RNA
For highly charged linear polyanions (such as nucleic acids), Manning's condensation theory demonstrates that whenever the linear charge spacing parameter $\xi$ exceeds a critical threshold:
$$\xi = \frac{e^2}{4 \pi \epsilon_0 \epsilon_r k_B T b} > 1.0$$
counter-ions will **spontaneously condense** onto the macromolecular surface, reducing the effective charge density until $\xi_{\text{eff}} = 1.0$.
In an RNA double helix ($b \approx 1.7\,\text{\AA}$ per phosphate), $\approx 76\%$ of the formal phosphate charge is counter-balanced by a condensed counter-ion sheath.

### 3. Physiological Salt Arithmetic ($150\,\text{mM } \text{NaCl}$)
Beyond basic neutralization, the bulk solution must match physiological ionic strength ($C_{\text{salt}} = 0.150\,\text{M}$):
1. **Neutralizing Counter-Ions:**  
   If the solute has net charge $Q_{\text{solute}} < 0$, add $N_{\text{neut}} = |Q_{\text{solute}}|$ sodium ions ($\text{Na}^+$).  
   If $Q_{\text{solute}} > 0$, add $N_{\text{neut}} = Q_{\text{solute}}$ chloride ions ($\text{Cl}^-$).
2. **Additional Salt Pairs:**  
   To achieve molar concentration $C = 0.150\,\text{M}$:
   $$N_{\text{pairs}} = \text{Round}\left( C_{\text{salt}} \times N_A \times V_{\text{water}} \right)$$
   where $V_{\text{water}} = N_{\text{water}} \times v_{\text{water}}$ (with $v_{\text{water}} \approx 30.0\,\text{\AA}^3 = 3.0 \times 10^{-26}\,\text{L}$ per molecule).

Let's implement a verified Python function calculating exact ion counts.


In [ ]:
# Rigorous salt and counter-ion arithmetic calculator
AVOGADRO = 6.02214076e23

def calculate_exact_ion_counts(net_charge: int, n_water_molecules: int, 
                               salt_conc_molar: float = 0.150) -> dict:
    """
    Computes exact counter-ion and co-ion counts to achieve net charge neutrality
    and target physiological ionic strength.
    """
    # Approximate volume of N water molecules at 300K, 1 bar (density ~ 0.997 g/cm^3)
    # Molar mass of water = 18.015 g/mol -> volume per water molecule ~ 30.0 Å^3 = 3.0e-26 L
    water_volume_liters = n_water_molecules * (18.015 / (0.997 * 1000.0 * AVOGADRO))
    
    # Calculate additional salt ion pairs needed for target ionic strength
    # N_pairs = C * V * N_A
    exact_pairs = salt_conc_molar * water_volume_liters * AVOGADRO
    n_pairs = int(round(exact_pairs))
    
    # Neutralizing counter-ions
    if net_charge < 0:
        # Solute is anionic -> add Na+ to neutralize
        n_na = n_pairs + abs(net_charge)
        n_cl = n_pairs
    elif net_charge > 0:
        # Solute is cationic -> add Cl- to neutralize
        n_na = n_pairs
        n_cl = n_pairs + net_charge
    else:
        # Solute is neutral
        n_na = n_pairs
        n_cl = n_pairs
        
    final_cell_charge = net_charge + (n_na * (+1)) + (n_cl * (-1))
    assert final_cell_charge == 0, "Error: Periodic cell is not electroneutral!"
    
    return {
        "solute_net_charge": net_charge,
        "n_water": n_water_molecules,
        "water_volume_liters": water_volume_liters,
        "salt_conc_M": salt_conc_molar,
        "n_added_salt_pairs": n_pairs,
        "total_Na_plus": n_na,
        "total_Cl_minus": n_cl,
        "final_net_charge": final_cell_charge
    }

# Case A: PETase Chain A at pH 7.4 (Net charge ≈ -2) in ~10,000 waters
petase_ions = calculate_exact_ion_counts(net_charge=-2, n_water_molecules=9850, salt_conc_molar=0.150)
print("=== PETase (Protein Thread) Ion Accounting ===")
for k, v in petase_ions.items():
    print(f"  {k}: {v if isinstance(v, (int, str)) else f'{v:.4e}'}")

# Case B: add A-riboswitch aptamer (71 nt -> Net charge ≈ -70) in ~12,000 waters
rna_ions = calculate_exact_ion_counts(net_charge=-70, n_water_molecules=12400, salt_conc_molar=0.150)
print("\n=== Purine Riboswitch (RNA Thread) Ion Accounting ===")
for k, v in rna_ions.items():
    print(f"  {k}: {v if isinstance(v, (int, str)) else f'{v:.4e}'}")


---
## Section 3: Periodic Boundary Conditions and Unit Cell Geometry

Macromolecular simulations are carried out in a **Periodic Boundary Condition (PBC)** unit cell. When an atom leaves the box on the right ($+x$), it instantly re-enters from the left ($-x$) with identical velocity, eliminating artificial vacuum boundary surfaces.

### Minimum Solute-to-Wall Clearance ($d_{\text{buffer}} \ge 1.0\,\text{nm}$)
In non-bonded force evaluations, electrostatic and van der Waals interactions are evaluated up to a spherical cutoff distance:
$$r_{\text{cutoff}} \approx 0.9 - 1.0\,\text{nm} \quad (9.0 - 10.0\,\text{\AA})$$

**The Minimum Image Convention:**  
An atom must never interact with its own periodic image or interact with the same neighbor twice.
To prevent self-interaction across periodic boundaries, the minimum distance from any solute atom to the box edge must be **at least the cutoff distance**:
$$d_{\text{buffer}} \ge r_{\text{cutoff}} = 1.0\,\text{nm} = 10.0\,\text{\AA}$$

Thus, for a solute with maximum Cartesian span $L_{\text{solute}} = \max(\Delta X, \Delta Y, \Delta Z)$, the periodic box edge length must be:
$$L_{\text{box}} \ge L_{\text{solute}} + 2 \times d_{\text{buffer}}$$

Let's compute the physical dimensions and required cubic box size for PETase Chain A.


In [ ]:
# Parse cleaned PETase Chain A coordinates to measure Cartesian span
import os
import requests
from Bio.PDB import PDBParser, PDBIO, Select
from pdbfixer import PDBFixer
from openmm import app

os.makedirs("data", exist_ok=True)
clean_chain_a_pdb = "data/6ane_chainA_clean.pdb"
input_pdb_path = "data/6ane.pdb"

# Self-containment guard: download and isolate Chain A with PDBFixer if missing
if not os.path.exists(clean_chain_a_pdb):
    if not os.path.exists(input_pdb_path):
        print("Downloading PDB 6ANE from RCSB...")
        try:
            res = requests.get("https://files.rcsb.org/download/6ane.pdb", timeout=15)
            res.raise_for_status()
            with open(input_pdb_path, "w") as f:
                f.write(res.text)
            print("Download complete.")
        except Exception as e:
            print(f"Notice: Network download notice: {e}")

    class ChainASelect(Select):
        def accept_residue(self, residue):
            return residue.get_parent().id == "A" and residue.id[0].strip() == ""

    if os.path.exists(input_pdb_path):
        parser = PDBParser(QUIET=True)
        struct = parser.get_structure("6ane", input_pdb_path)
        io = PDBIO()
        io.set_structure(struct)
        io.save(clean_chain_a_pdb, ChainASelect())
        print("Chain A extracted to:", clean_chain_a_pdb)

    # Heal terminal OXT using PDBFixer to prevent OpenMM template errors
    print("Applying PDBFixer to ensure terminal template integrity...")
    fixer = PDBFixer(filename=clean_chain_a_pdb)
    fixer.findMissingResidues()
    fixer.missingResidues = {}
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    with open(clean_chain_a_pdb, "w") as f:
        app.PDBFile.writeFile(fixer.topology, fixer.positions, f)
    print("Terminal OXT successfully added to Chain A.")

parser = PDBParser(QUIET=True)
structure = parser.get_structure("petase", clean_chain_a_pdb)[0]["A"]
coords = np.array([atom.get_coord() for atom in structure.get_atoms()])

min_coords = np.min(coords, axis=0)
max_coords = np.max(coords, axis=0)
spans = max_coords - min_coords
max_span = np.max(spans)

BUFFER_PADDING_ANGSTROM = 10.0  # 1.0 nm
box_length_angstrom = max_span + 2.0 * BUFFER_PADDING_ANGSTROM
box_volume_angstrom3 = box_length_angstrom ** 3

print("=== Solute Geometry & Periodic Box Dimensions ===")
print(f"PETase Cartesian Spans: ΔX = {spans[0]:.2f} Å | ΔY = {spans[1]:.2f} Å | ΔZ = {spans[2]:.2f} Å")
print(f"Maximum Solute Extent:  {max_span:.2f} Å")
print(f"Minimum Periodic Buffer: 2 x {BUFFER_PADDING_ANGSTROM:.1f} Å = {2*BUFFER_PADDING_ANGSTROM:.1f} Å")
print(f"Required Cubic Box Side: L_box = {box_length_angstrom:.2f} Å ({box_length_angstrom / 10.0:.2f} nm)")
print(f"Estimated Unit Cell Volume: {box_volume_angstrom3:,.1f} Å³")


---
## Section 4: Production-Grade Explicit Solvation and Relaxation in OpenMM

We now execute the full explicit-solvent pipeline in OpenMM:
1. Load cleaned protein coordinates.
2. Parameterize with the **Amber14-SB** protein force field and **TIP3P** water (`amber14/tip3p.xml`).
3. Add missing hydrogens at physiological pH $7.4$.
4. Solvate the protein in a cubic box with $1.0\,\text{nm}$ buffer padding.
5. Add neutralizing $\text{Na}^+$ counter-ions and bring the solution to $150\,\text{mM } \text{NaCl}$.
6. Execute Steepest Descent + L-BFGS energy minimization to relieve solvent-solute steric clashes.


In [ ]:
def get_optimal_openmm_platform():
    """Finds the fastest available hardware platform (CUDA -> OpenCL -> CPU)."""
    for p_name in ['CUDA', 'OpenCL', 'CPU']:
        try:
            plat = mm.Platform.getPlatformByName(p_name)
            if p_name == 'CUDA':
                test_ctx = mm.Context(mm.System(), mm.VerletIntegrator(0.001), plat)
                del test_ctx
            return plat, p_name
        except Exception:
            continue
    return mm.Platform.getPlatformByName('Reference'), 'Reference'

platform, platform_name = get_optimal_openmm_platform()
print(f"OpenMM Hardware Platform Configured: {platform_name}")

print("Building explicit solvent system with OpenMM...")

# Ensure C-terminal OXT atom exists so Amber force field template matches without error
fixer = PDBFixer(filename=clean_chain_a_pdb)
fixer.findMissingResidues()
fixer.missingResidues = {}
fixer.findMissingAtoms()
fixer.addMissingAtoms()
with open(clean_chain_a_pdb, "w") as f:
    app.PDBFile.writeFile(fixer.topology, fixer.positions, f)

# Load PDB and Force Field
pdb = app.PDBFile(clean_chain_a_pdb)
# Force field combination: Amber14-SB (protein) + TIP3P (water and ions)
forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3p.xml')

# Modeller setup
modeller = app.Modeller(pdb.topology, pdb.positions)
# Add hydrogens at pH 7.4
modeller.addHydrogens(forcefield, pH=7.4)

# Add explicit water box with 1.0 nm (10.0 Å) buffer padding and 150 mM NaCl
print("Adding TIP3P water box and 150 mM NaCl ions...")
modeller.addSolvent(
    forcefield,
    model='tip3p',
    padding=1.0 * unit.nanometer,
    ionicStrength=0.150 * unit.molar,
    neutralize=True
)

# Extract atom and residue counts
n_total_atoms = modeller.topology.getNumAtoms()
n_residues = modeller.topology.getNumResidues()

print(f"Solvation Complete:")
print(f"  Total Atoms in Periodic Unit Cell: {n_total_atoms:,}")
print(f"  Total Residues (Protein + Water + Ions): {n_residues:,}")

# Create OpenMM System with Periodic Boundary Conditions and PME
# Cutoff at 1.0 nm (10 Å)
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.0 * unit.nanometer,
    constraints=app.HBonds
)

# Set up integrator and simulation context
integrator = mm.LangevinMiddleIntegrator(300*unit.kelvin, 1.0/unit.picoseconds, 0.002*unit.picoseconds)
simulation = app.Simulation(modeller.topology, system, integrator, platform)
simulation.context.setPositions(modeller.positions)

# Evaluate initial potential energy before minimization
state_unmin = simulation.context.getState(getEnergy=True)
e_unmin = state_unmin.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
print(f"\nInitial Potential Energy (With Clashing Explicit Waters): {e_unmin:,.2f} kJ/mol")

# Run L-BFGS Energy Minimization
print("Running L-BFGS Energy Minimization in Explicit Solvent...")
simulation.minimizeEnergy(tolerance=10.0*unit.kilojoules_per_mole/unit.nanometer, maxIterations=300)

state_min = simulation.context.getState(getEnergy=True, getPositions=True)
e_min = state_min.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
delta_E = e_min - e_unmin

print(f"Final Potential Energy (After Relaxation):              {e_min:,.2f} kJ/mol")
print(f"Total Potential Energy Relieved:                         {delta_E:,.2f} kJ/mol")

# Save solvated and relaxed system to PDB
solvated_pdb_path = "data/6ane_solvated_minimized.pdb"
with open(solvated_pdb_path, "w") as f:
    app.PDBFile.writeFile(simulation.topology, state_min.getPositions(), f)

print(f"\nSolvated, neutralized, minimized system saved to: {solvated_pdb_path}")


### System Composition Verification
Let's verify the exact molecular breakdown (protein, water, sodium ions, chloride ions) of our solvated simulation system.


In [ ]:
# Parse topology to count species
counts = {"Protein": 0, "Water (HOH)": 0, "Sodium (NA)": 0, "Chloride (CL)": 0}

for res in modeller.topology.residues():
    rname = res.name.strip().upper()
    if rname in ["HOH", "WAT", "TIP3"]:
        counts["Water (HOH)"] += 1
    elif rname in ["NA", "SOD"]:
        counts["Sodium (NA)"] += 1
    elif rname in ["CL", "CLA"]:
        counts["Chloride (CL)"] += 1
    else:
        counts["Protein"] += 1

comp_df = pd.DataFrame([{"Component": k, "Count": v} for k, v in counts.items()])
comp_df["Atoms"] = [
    counts["Protein"] * 15,  # Approx 15 atoms per residue average
    counts["Water (HOH)"] * 3,
    counts["Sodium (NA)"],
    counts["Chloride (CL)"]
]

print("Table 2.2: Solvated System Inventory Breakdown")
display(comp_df)


---
## Section 5: 3D Visualization of Solvated Systems in py3Dmol

Let's visualize the solvated unit cell in `py3Dmol`:
1. Protein core rendered as a cyan cartoon.
2. Active-site catalytic triad ($	ext{Ser133}$–$	ext{Asp179}$–$	ext{His210}$) and $	ext{Trp158}$ rendered as bright sticks.
3. First hydration shell waters (within $4.0\,\text{\AA}$ of the protein) rendered as delicate red spheres or lines.
4. Sodium ($\text{Na}^+$) and Chloride ($\text{Cl}^-$) ions rendered as blue and yellow van der Waals spheres.


In [ ]:
def render_solvated_active_site(solvated_pdb: str):
    """Visualizes solvated protein with first hydration shell and counter-ions."""
    view = py3Dmol.view(width=720, height=480)
    with open(solvated_pdb, "r") as f:
        view.addModel(f.read(), "pdb")
        
    # 1. Protein backbone
    view.setStyle({"protein": True}, {"cartoon": {"color": "#00cccc", "opacity": 0.85}})
    
    # 2. Catalytic triad and Trp158 sticks
    view.addStyle({"resi": ["133", "179", "210", "158"]}, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.22}})
    
    # 3. First hydration shell waters: within 4.0 Å of catalytic Ser133
    view.addStyle({"resn": "HOH", "within": {"distance": 4.5, "sel": {"resi": "133"}}}, 
                  {"stick": {"colorscheme": "redCarbon", "radius": 0.15}})
    
    # 4. Sodium ions as blue spheres, Chloride ions as yellow spheres
    view.addStyle({"resn": "NA"}, {"sphere": {"color": "#3366ff", "radius": 1.2}})
    view.addStyle({"resn": "CL"}, {"sphere": {"color": "#ffcc00", "radius": 1.4}})
    
    view.zoomTo({"resi": "133"})
    return view

solv_view = render_solvated_active_site(solvated_pdb_path)
solv_view.show()


### Formative Assessment: Periodic Boundary Conditions & PME
Test your understanding of boundary conditions and neutralization.


In [ ]:
quiz_section_2 = [
    {
        "question": "What happens if an explicit solvent molecular dynamics simulation is initiated with a buffer padding of only 0.4 nm (4.0 Å) between the protein and the periodic box wall?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "The minimum image convention is violated; the protein will interact directly with its own periodic image across the boundary, creating severe unphysical self-interaction artifacts.",
                "correct": True,
                "feedback": "Correct! Because non-bonded cutoffs are typically 0.9-1.0 nm, a buffer padding of >= 1.0 nm is mandatory to prevent an atom from seeing its own periodic replica."
            },
            {
                "answer": "The simulation runs faster without any loss of accuracy.",
                "correct": False,
                "feedback": "Incorrect. While smaller boxes have fewer atoms, the physical validity is destroyed by self-interaction artifacts."
            },
            {
                "answer": "The protein will immediately undergo spontaneous peptide bond cleavage.",
                "correct": False,
                "feedback": "Incorrect. Classical force fields maintain fixed covalent bonds."
            },
            {
                "answer": "Water molecules will turn into gas vapor.",
                "correct": False,
                "feedback": "Incorrect. Barostats maintain 1 bar liquid density."
            }
        ]
    },
    {
        "question": "Why is it mandatory for the net electrical charge of a periodic simulation cell to equal exactly 0.0 under Particle Mesh Ewald (PME) electrostatics?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "A non-neutral cell under infinite periodic lattice summation causes the reciprocal-space electrostatic energy term to diverge toward infinity unless an artificial uniform neutralizing background plasma is applied, which introduces artifactual conformational forces.",
                "correct": True,
                "feedback": "Correct! Neutralizing counter-ions eliminate infinite lattice energy divergence in Ewald summation."
            },
            {
                "answer": "Because ions are toxic to computer GPUs.",
                "correct": False,
                "feedback": "Incorrect. GPUs calculate ion positions and Coulomb interactions routinely."
            },
            {
                "answer": "To force all amino acids to carry a formal charge of zero.",
                "correct": False,
                "feedback": "Incorrect. Individual amino acids retain their assigned partial and formal charges; the sum over the entire cell is zero."
            },
            {
                "answer": "Because PDB files cannot store more than 10 sodium atoms.",
                "correct": False,
                "feedback": "Incorrect. Coordinate files can store arbitrary numbers of heteroatoms."
            }
        ]
    }
]

display_quiz(quiz_section_2)


---
## Section 6: Open-Ended Investigation & Lab Summary

### Open-Ended Challenge Questions (For Graded Laboratory Report)
1. **Solvation Box Shape Optimization (Cubic vs. Rhombic Dodecahedron):**  
   A cubic box is straightforward to construct, but wastes solvent volume in the corners. A **rhombic dodecahedron** unit cell fills 3D space with the closest sphere packing (Kepler's conjecture). For a roughly spherical globular protein like PETase, switching from a cube to a rhombic dodecahedron reduces the required solvent volume by approximately **$29\%$**. Calculate the reduction in total atom count and predict the percentage increase in computational MD throughput (ns/day) achieved by this geometric box choice.
   
2. **Magnesium Condensation on RNA Riboswitch:**  
   Solvate the purine riboswitch (PDB `1Y26`, 71 nt, charge $\approx -70\,e$) using OpenMM. Replace monovalent $\text{Na}^+$ ions with divalent $\text{Mg}^{2+}$ ions ($+2\,e$). How many $\text{Mg}^{2+}$ ions are required for electroneutrality compared to $\text{Na}^+$? After energy minimization, measure the distance between the closest $\text{Mg}^{2+}$ ion and the active-site adenine binding pocket. Does divalent magnesium coordinate preferentially to non-bridging phosphate oxygens?

---

### Key Takeaways from Lab 2.1
- **Solvation Physics:** Implicit solvent enables fast sampling; explicit water (TIP3P, TIP4P-Ew, OPC) is required for realistic viscosity, bridging hydrogen bonds, and hydrophobic dewetting.
- **Electroneutrality:** PME electrostatics strictly requires unit cell net charge $Q_{\text{net}} = 0$.
- **Physiological Atmosphere:** Systems must be brought to $150\,\text{mM } \text{NaCl}$ to reproduce cellular screening.
- **Minimum Image Convention:** Periodic buffer padding must satisfy $d_{\text{buffer}} \ge r_{\text{cutoff}} = 1.0\,\text{nm}$ to prevent unphysical self-interaction across periodic boundary walls.

You are now prepared to advance to **Lab 2.2: Molecular Dynamics Simulation, Thermostats, Barostats, and Trajectory Analysis**!


In [ ]:
# Laboratory execution lockfile manifest
import json

lab2_1_manifest = {
    "lab_id": "Lab_2.1_Solvation_Ions_MD_Preparation",
    "course_seed": COURSE_SEED,
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "openmm_version": mm.__version__,
    "biopython_version": "1.83",
    "protein_target": "6ANE_PETase_ChainA",
    "solute_max_span_angstrom": round(float(max_span), 2),
    "box_buffer_padding_nm": 1.0,
    "target_salt_molar": 0.150,
    "solvated_total_atoms": int(n_total_atoms),
    "solvated_water_count": int(counts["Water (HOH)"]),
    "added_na_ions": int(counts["Sodium (NA)"]),
    "added_cl_ions": int(counts["Chloride (CL)"]),
    "initial_unminimized_energy_kJ_mol": round(float(e_unmin), 2),
    "final_minimized_energy_kJ_mol": round(float(e_min), 2),
    "energy_relief_kJ_mol": round(float(delta_E), 2)
}

manifest_path = "manifests/lab2_1_lockfile.json"
with open(manifest_path, "w") as f:
    json.dump(lab2_1_manifest, f, indent=4)

print(f"Manifest written successfully to: {manifest_path}")
print(json.dumps(lab2_1_manifest, indent=2))
